## Build neighbor traversal dataset

### step 1 - create nearest road dataset
1. Connect to osm database
1. Load resolution 3 cell center data
1. Run nearest road query for each cell center
1. Save data

### build resolution 3 road snapped dataset
- load all cell center points for resolution 3 hexagons
- put into dataframe

| Cell ID         | resolution | Center lat | Center lon  | Roadsnap lat | Roadsnap lon | Distance |
| --------------- | ---------- | ---------- | ----------- | ------------ | ------------ | -------- |
| 8312cafffffffff | 3          | 49.003713  | -114.769397 | 48.998919    | -114.7706613 | 541      |
| ... |||||||


In [1]:
import h3
from h3 import LatLngPoly, LatLngMultiPoly
import json
from shapely.geometry import shape
from shapely.ops import unary_union
from sqlalchemy import create_engine, text
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
import folium
from config import DATABASE_URL
resolution = 3
engine = create_engine(DATABASE_URL)

def draw_conus_polygon():
    with open("datasets/conus-states.json", "r") as f:
        state_geo = json.load(f)

    states = [shape(f["geometry"]) for f in state_geo["features"]]
    conus = unary_union(states)
    return conus

def convert_polygon_to_h3(geom):
    if geom.geom_type == "Polygon":
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in geom.interiors
        ]
        return LatLngPoly(exterior, holes)

    elif geom.geom_type == "MultiPolygon":
        return LatLngMultiPoly(*(convert_polygon_to_h3(p) for p in geom.geoms))

def initialize_cell_centers_dataframe(cells):
    rows = [
        (cell, *h3.cell_to_latlng(cell))
        for cell in cells
    ]

    df = pd.DataFrame(rows, columns=["CellID", "Latitude", "Longitude"])

    df["Latitude"] = df["Latitude"].round(6)
    df["Longitude"] = df["Longitude"].round(6)
    return df

def get_road_snapped_point(lon, lat):
    
    sql = text("""
        SELECT
          ST_Y(geom_4326) AS lat,
          ST_X(geom_4326) AS lon
        FROM (
          SELECT
            ST_Transform(
              ST_ClosestPoint(way, input.geom),
              4326
            ) AS geom_4326
          FROM planet_osm_roads
          CROSS JOIN (
            SELECT ST_Transform(
                    ST_SetSRID(ST_Point(:lon, :lat), 4326),
                    3857
                  ) AS geom
          ) AS input
          WHERE highway IS NOT NULL
          ORDER BY way <-> input.geom
          LIMIT 1
        ) snapped;
    """) 
    
    with engine.connect() as conn:
      result = conn.execute(sql, {"lon": lon, "lat": lat}).fetchone()

    rslat = round(result[0], 6)
    rslon = round(result[1], 6)

    return rslat, rslon

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return round(R * c)

def add_road_snapped_data_to_dataframe(df):
    df[["RSLatitude", "RSLongitude"]] = df.apply(
        lambda row: get_road_snapped_point(row["Longitude"], row["Latitude"]), 
        axis=1, 
        result_type="expand"
    )

    df["RSDistance"] = df.apply(
        lambda row: haversine(
            row["Latitude"],
            row["Longitude"],
            row["RSLatitude"],
            row["RSLongitude"]
        ),
        axis=1
    )

    # check if rs lat/lon is in same cellID
    df["RSCellID"] = df.apply(
        lambda row : h3.latlng_to_cell(
            row["RSLatitude"], 
            row["RSLongitude"], 
            resolution
            ),
            axis=1
    )

    df["ValidSnap"] = df["CellID"] == df["RSCellID"]   
    return df    

conus = draw_conus_polygon()
h3_conus = convert_polygon_to_h3(conus)
h3_cells = h3.polygon_to_cells_experimental(
    h3shape=h3_conus, 
    res=resolution, 
    contain="overlap"
)
df = initialize_cell_centers_dataframe(h3_cells)
df = add_road_snapped_data_to_dataframe(df)

In [2]:
# Plot data

m = folium.Map(location=(45, -115), zoom_start=7)

for index, row in df.iterrows():


    if row["ValidSnap"] == True:
        color = "green"
        outline = "black"

    else:
        color = "red"
        outline = "red"
    
    folium.CircleMarker(
        location=[row["RSLatitude"], row["RSLongitude"]],
        radius=3,
        fill=True,
        fill_opacity=1,
        color=color
    ).add_to(m)

    hexagon = h3.cell_to_boundary(row["CellID"])

    folium.Polygon(
        locations=hexagon,
        weight=1,
        color="black",
        fill=False
    ).add_to(m)

    folium.PolyLine(
        locations=[
            [row["Latitude"], row["Longitude"]],
            [row["RSLatitude"], row["RSLongitude"]]
            ],
        color=color,
        weight=2
    ).add_to(m)

    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=3,
        fill=True,
        fill_opacity=1,
        color="blue"
    ).add_to(m)
m

### step 2 - create cell to neighbors transit dataset
1. for each cell - identify neighbors
1. for each neighbor 
    - find transit distance

### for each cell identify valid neighbors

In [ ]:
df["Neighbors"] = df["CellID"].apply(lambda x: h3.grid_ring(x, 1))

valid_cells = set(df["CellID"])

def id_valid_neighbors(neighbors: list[str]) -> list[str]:
    return [h for h in neighbors if h in valid_cells]


df["ValidNeighbors"] = df["Neighbors"].apply(id_valid_neighbors)

df.head(10)

In [ ]:
### test plot ###
df = df.loc[df["CellID"] == "852600cbfffffff"]
df = df.reset_index()
lst = df.loc[0,"Neighbors"]

# print(df["Neighbors"])
# print(type(df["Neighbors"]))
lst

### for each valid neighbor, find transit distance

### create valid neighbor transit distance dict



## 